# Lille — Co-visiting et Node2Vec : réévaluation des liens existants

**Objectif :** construire la matrice de probabilités de co-visiting, puis utiliser Node2Vec pour **réajuster uniquement les relations observées en 2013**. Les liens absents restent nuls.

- **2013** : construction des matrices et entraînement de Node2Vec sur le graphe orienté et pondéré.
- **2014** : choix de la force de réévaluation `lambda`.
- **2015** : comparaison finale indépendante.

Deux métriques simples : **probabilité moyenne pondérée des relations observées** et **Top-5 pondéré**. Pas de log-loss ni de lissage. Les données représentent des co-visites agrégées, pas des déplacements successifs certifiés.

## 1 — Charger automatiquement le CSV

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

POIDS = "NbPerMaxDurationDays_14"
K = 5
NOM_CSV = "MEL_circulationGraph_0.csv"
# Si tu connais son emplacement, indique-le ici (sinon laisse None) :
CHEMIN_CSV_MANUEL = None

# D'abord les emplacements les plus probables, puis les disques Windows si besoin.
def trouver_csv(nom):
    if CHEMIN_CSV_MANUEL:
        p = Path(CHEMIN_CSV_MANUEL).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f"Chemin manuel incorrect : {p}")
        return p
    racines = [Path.cwd(), Path.home(), Path.home() / 'Downloads',
               Path.home() / 'Documents', Path.home() / 'Desktop',
               Path.home() / 'OneDrive']
    try:
        racines.append(Path(__file__).parent)  # disponible hors Jupyter
    except NameError:
        pass
    for racine in racines:
        candidat = racine / nom
        if candidat.is_file():
            return candidat
    # Recherche récursive, arrêt dès la première correspondance.
    # Peut être longue si le fichier est sur un disque entier.
    disques = [Path(f'{lettre}:/') for lettre in 'CDEFGHIJKLMNOPQRSTUVWXYZ'
               if Path(f'{lettre}:/').exists()] if os.name == 'nt' else []
    for racine in [Path.cwd(), Path.home(), *disques]:
        if not racine.exists():
            continue
        for dossier, sous_dossiers, fichiers in os.walk(racine, onerror=lambda e: None):
            sous_dossiers[:] = [x for x in sous_dossiers
                                  if x.lower() not in {'$recycle.bin', 'system volume information',
                                                       '.git', 'node_modules', '.venv', 'venv'}]
            if nom in fichiers:
                return Path(dossier) / nom
    raise FileNotFoundError(f"Impossible de trouver {nom}. Renseigne CHEMIN_CSV_MANUEL.")

chemin = trouver_csv(NOM_CSV)
print('CSV trouvé :', chemin)
donnees = pd.read_csv(chemin, sep=';')
donnees = donnees.loc[donnees[POIDS] > 0].copy()
donnees = donnees.loc[donnees.gid_from != donnees.gid_to].copy()
print('Lignes après filtrage :', len(donnees))
print(donnees.year.value_counts().sort_index().loc[2013:2015])
DOSSIER_RESULTATS = Path('resultats_matrices_simples')
DOSSIER_RESULTATS.mkdir(exist_ok=True)


CSV trouvé : C:\Users\aurel\OneDrive\Année 4\Parcours recherche\GitHub\MEL_circulationGraph_0.csv
Lignes après filtrage : 61073
year
2013    1947
2014    3623
2015    7270
Name: count, dtype: int64


## 2 — Matrice initiale de co-visiting et probabilités (2013)

Chaque ligne est un POI de départ ; chaque colonne, un POI d’arrivée. On divise les poids d’une ligne par leur somme. Pour un POI sans relation sortante, on conserve le même repli par popularité dans les deux modèles.

In [2]:
train = donnees.loc[donnees.year == 2013].copy()
validation = donnees.loc[donnees.year == 2014].copy()
test = donnees.loc[donnees.year == 2015].copy()

relations_2013 = train.groupby(['gid_from','gid_to'], as_index=False)[POIDS].sum()
lieux = sorted(set(relations_2013.gid_from) | set(relations_2013.gid_to))
matrice_co = pd.DataFrame(0.0, index=lieux, columns=lieux)
for source, destination, poids in relations_2013.itertuples(index=False, name=None):
    matrice_co.at[source, destination] = float(poids)

W = matrice_co.to_numpy(copy=True)
np.fill_diagonal(W, 0.0)
popularite = W.sum(axis=0)
popularite = popularite / popularite.sum()

def normaliser_lignes(scores):
    scores = np.asarray(scores, dtype=float).copy()
    np.fill_diagonal(scores, 0.0)
    sommes = scores.sum(axis=1, keepdims=True)
    resultat = np.divide(scores, sommes, out=np.zeros_like(scores), where=sommes > 0)
    # Sources sans sorties : destinations populaires, sans auto-recommandation.
    for i in np.where(sommes[:, 0] == 0)[0]:
        ligne = popularite.copy()
        ligne[i] = 0
        resultat[i] = ligne / ligne.sum() if ligne.sum() > 0 else 0
    return resultat

P_co = normaliser_lignes(W)
print('Matrice :', P_co.shape, '| Lignes 2013 :', len(train))
print('Sommes des lignes :', P_co.sum(axis=1).min(), P_co.sum(axis=1).max())

# Affichage limité aux 10 premiers lieux pour rester lisible dans VS Code.
print("Matrice des poids co-visiting (extrait 10 x 10) :")
display(matrice_co.iloc[:10, :10])
print("Matrice des probabilités co-visiting (extrait 10 x 10) :")
display(pd.DataFrame(P_co, index=lieux, columns=lieux).iloc[:10, :10].round(4))


Matrice : (594, 594) | Lignes 2013 : 1947
Sommes des lignes : 0.9999999999999998 1.0
Matrice des poids co-visiting (extrait 10 x 10) :


,196911,196912,196913,196916,196917,196920,196928,196931,209260,209264
196911,0.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
196912,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
196913,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196916,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
196917,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
196920,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196928,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196931,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
209260,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
209264,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Matrice des probabilités co-visiting (extrait 10 x 10) :


,196911,196912,196913,196916,196917,196920,196928,196931,209260,209264
196911,0.0,0.08,0.0,0.0,0.04,0.0400,0.0,0.0,0.0000,0.0000
196912,0.0,0.00,0.0,0.0,0.00,0.0476,0.0,0.0,0.0000,0.0000
196913,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
196916,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.1429,0.0000
196917,0.0,0.00,0.0,0.0,0.00,0.0588,0.0,0.0,0.0000,0.0000
196920,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
196928,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
196931,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
209260,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.1667
209264,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000


## 3 — Node2Vec classique sur le graphe orienté (2013)

Si besoin, exécute dans une cellule séparée : `%pip install node2vec setuptools`. Les marches aléatoires utilisent les poids des relations et leur direction. Les embeddings ont 64 dimensions.

In [3]:
import networkx as nx
from node2vec import Node2Vec
from sklearn.metrics.pairwise import cosine_similarity

G = nx.DiGraph()
G.add_nodes_from(lieux)
for source, destination, poids in relations_2013.itertuples(index=False, name=None):
    G.add_edge(source, destination, weight=float(poids))

print('Graphe orienté :', G.is_directed(), '| Arêtes :', G.number_of_edges())
n2v = Node2Vec(G, dimensions=64, walk_length=20, num_walks=50,
              p=1, q=1, weight_key='weight', workers=1, seed=42, quiet=True)
modele = n2v.fit(window=5, min_count=1, batch_words=128, seed=42)
embeddings = np.vstack([modele.wv[str(lieu)] for lieu in lieux])
print('Embeddings :', embeddings.shape)


c:\Users\aurel\.venv312\Lib\site-packages\node2vec\edges.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\aurel\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Graphe orienté : True | Arêtes : 1818
Embeddings : (594, 64)


## 4 — Réévaluer **uniquement** les liens existants

Pour chaque lien observé en 2013, on ajuste son poids :

\[w'_{ij}=w_{ij}(1+\lambda s_{ij})\]

avec \(s_{ij}\) la similarité cosinus Node2Vec ramenée entre 0 et 1. On normalise ensuite chaque ligne pour obtenir des probabilités. **Si \(w_{ij}=0\), alors \(w'_{ij}=0\)**. La structure du graphe reste donc inchangée. La similarité cosinus est symétrique, mais les poids et probabilités de co-visiting restent dirigés.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

similarites = cosine_similarity(embeddings)
scores_n2v = (np.clip(similarites, -1, 1) + 1) / 2
np.fill_diagonal(scores_n2v, 0.0)

# La multiplication par W protège les liens absents : 0 reste 0.
def construire_hybride(lam):
    W_ajuste = W * (1 + lam * scores_n2v)
    return normaliser_lignes(W_ajuste)

LAMBDA_EXEMPLE = 1.0
P_hybride_exemple = construire_hybride(LAMBDA_EXEMPLE)
assert np.allclose(construire_hybride(0), P_co)
assert np.allclose(P_hybride_exemple.sum(axis=1), 1)
assert np.all(P_hybride_exemple[W == 0] == P_co[W == 0])  # même repli pour les sources sans sortie

print('Matrice Node2Vec (similarités, extrait 10 x 10) :')
display(pd.DataFrame(scores_n2v, index=lieux, columns=lieux).iloc[:10, :10].round(4))
print('Matrice hybride (lambda=1, extrait 10 x 10) :')
display(pd.DataFrame(P_hybride_exemple, index=lieux, columns=lieux).iloc[:10, :10].round(4))
print('Différence maximale avec la matrice initiale :',
      np.max(np.abs(P_hybride_exemple - P_co)))


Matrice Node2Vec (similarités, extrait 10 x 10) :


,196911,196912,196913,196916,196917,196920,196928,196931,209260,209264
196911,0.0000,0.8868,0.6032,0.8048,0.9022,0.8805,0.5889,0.5588,0.7530,0.8660
196912,0.8868,0.0000,0.6700,0.8848,0.8296,0.9463,0.6708,0.6454,0.8538,0.9594
196913,0.6032,0.6700,0.0000,0.7761,0.6028,0.6504,0.6446,0.8999,0.7153,0.6761
196916,0.8048,0.8848,0.7761,0.0000,0.7983,0.8702,0.6458,0.7476,0.8702,0.9030
196917,0.9022,0.8296,0.6028,0.7983,0.0000,0.8843,0.5634,0.5456,0.7865,0.8408
196920,0.8805,0.9463,0.6504,0.8702,0.8843,0.0000,0.6450,0.6316,0.8700,0.9328
196928,0.5889,0.6708,0.6446,0.6458,0.5634,0.6450,0.0000,0.7347,0.6334,0.6689
196931,0.5588,0.6454,0.8999,0.7476,0.5456,0.6316,0.7347,0.0000,0.7148,0.6568
209260,0.7530,0.8538,0.7153,0.8702,0.7865,0.8700,0.6334,0.7148,0.0000,0.9282
209264,0.8660,0.9594,0.6761,0.9030,0.8408,0.9328,0.6689,0.6568,0.9282,0.0000


Matrice hybride (lambda=1, extrait 10 x 10) :


,196911,196912,196913,196916,196917,196920,196928,196931,209260,209264
196911,0.0,0.0829,0.0,0.0,0.0418,0.0413,0.0,0.0,0.0000,0.0000
196912,0.0,0.0000,0.0,0.0,0.0000,0.0498,0.0,0.0,0.0000,0.0000
196913,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
196916,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.1445,0.0000
196917,0.0,0.0000,0.0,0.0,0.0000,0.0614,0.0,0.0,0.0000,0.0000
196920,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
196928,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
196931,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
209260,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.1718
209264,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000


Différence maximale avec la matrice initiale : 0.034446940990598196


## 5 — Visualiser un POI et ses destinations

Pour mieux voir la réévaluation, on affiche les **10 destinations historiques les plus fréquentes** d’un POI. On compare directement leurs probabilités avant/après Node2Vec.

In [5]:
# POI ayant le plus grand nombre de relations sortantes en 2013
idx_exemple = int(np.argmax((W > 0).sum(axis=1)))
lieu_exemple = lieux[idx_exemple]
voisins = np.where(W[idx_exemple] > 0)[0]
voisins_tries = voisins[np.argsort(-W[idx_exemple, voisins])][:10]

comparaison_poi = pd.DataFrame({
    'destination': [lieux[j] for j in voisins_tries],
    'co_visites_2013': W[idx_exemple, voisins_tries].astype(int),
    'proba_co_visiting': P_co[idx_exemple, voisins_tries],
    'similarite_node2vec': scores_n2v[idx_exemple, voisins_tries],
    'proba_hybride_lambda_1': P_hybride_exemple[idx_exemple, voisins_tries],
})
print('POI de départ :', lieu_exemple)
display(comparaison_poi.round(4))


POI de départ : 269820


,destination,co_visites_2013,proba_co_visiting,similarite_node2vec,proba_hybride_lambda_1
0,289349,5,0.0685,0.8294,0.0708
1,628635,4,0.0548,0.8999,0.0588
2,734546,4,0.0548,0.8863,0.0584
3,793722,3,0.0411,0.7667,0.0410
4,801264,3,0.0411,0.7945,0.0417
5,1489743,3,0.0411,0.7293,0.0401
6,1056055,3,0.0411,0.7323,0.0402
7,1026189,3,0.0411,0.7950,0.0417
8,2273093,3,0.0411,0.7095,0.0397
9,2294762,3,0.0411,0.6764,0.0389


## 6 — Évaluer sur les relations observées

On ne compare que les paires dont les deux POI existaient en 2013. La probabilité moyenne est pondérée par la fréquence de la relation dans l’année évaluée. On mesure aussi le Top-5 pondéré.

In [6]:
position = {lieu: i for i, lieu in enumerate(lieux)}

def evaluer(P, observations):
    connues = observations.loc[
        observations.gid_from.isin(position) & observations.gid_to.isin(position)
    ].copy()
    if connues.empty:
        raise ValueError('Aucune relation évaluable avec les lieux de 2013.')
    indices_sources = connues.gid_from.map(position).to_numpy()
    indices_destinations = connues.gid_to.map(position).to_numpy()
    poids = connues[POIDS].to_numpy(dtype=float)
    proba = P[indices_sources, indices_destinations]
    top5 = np.argsort(-P, axis=1, kind='stable')[:, :K]
    bonnes = (top5[indices_sources] == indices_destinations[:, None]).any(axis=1)
    return {
        'proba_moyenne': float(np.average(proba, weights=poids)),
        'top5': float(np.average(bonnes, weights=poids)),
        'relations_evaluees': len(connues),
        'part_poids_evaluee': float(poids.sum() / observations[POIDS].sum())
    }

print('Co-visiting sur 2014 :', evaluer(P_co, validation))
print('Hybride lambda=1 sur 2014 :', evaluer(P_hybride_exemple, validation))


Co-visiting sur 2014 : {'proba_moyenne': 0.042710371050866704, 'top5': 0.16950757575757575, 'relations_evaluees': 2748, 'part_poids_evaluee': 0.7745721271393643}
Hybride lambda=1 sur 2014 : {'proba_moyenne': 0.04310359934308468, 'top5': 0.1606691919191919, 'relations_evaluees': 2748, 'part_poids_evaluee': 0.7745721271393643}


## 7 — Choisir la force de réévaluation sur 2014

`lambda = 0` donne exactement la baseline. Une valeur plus grande renforce l’effet Node2Vec **sur les liens existants uniquement**. On choisit la valeur maximisant la probabilité moyenne pondérée sur 2014, sans utiliser 2015 pour le choix.

In [10]:
lambdas = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0,1.0,2.0]
lignes = []
for lam in lambdas:
    lignes.append({'lambda': lam, **evaluer(construire_hybride(lam), validation)})
resultats_validation = pd.DataFrame(lignes)
display(resultats_validation.round(5))
LAMBDA_CHOISI = float(resultats_validation.sort_values(
    ['proba_moyenne', 'lambda'], ascending=[False, True]
).iloc[0]['lambda'])
print('Lambda choisi sur 2014 :', LAMBDA_CHOISI)


,lambda,proba_moyenne,top5,relations_evaluees,part_poids_evaluee
0,0.00,0.04271,0.16951,2748,0.77457
1,0.25,0.04286,0.16067,2748,0.77457
2,0.50,0.04296,0.16067,2748,0.77457
3,1.00,0.04310,0.16067,2748,0.77457
4,2.00,0.04325,0.16067,2748,0.77457
5,4.00,0.04338,0.16067,2748,0.77457
6,8.00,0.04347,0.16067,2748,0.77457
7,1.00,0.04310,0.16067,2748,0.77457
8,2.00,0.04325,0.16067,2748,0.77457


Lambda choisi sur 2014 : 8.0


## 8 — Comparaison finale sur 2015

On utilise la même année et les mêmes relations pour les deux matrices. Si `lambda = 0` est choisi, les résultats seront identiques : c’est une conclusion expérimentale possible, pas un bug.

In [8]:
P_hybride_final = construire_hybride(LAMBDA_CHOISI)
comparaison = pd.DataFrame([
    {'modele': 'Co-visiting seul', **evaluer(P_co, test)},
    {'modele': 'Co-visiting + Node2Vec (liens existants)', **evaluer(P_hybride_final, test)}
])
display(comparaison)
print('Probabilité moyenne :', [f'{x:.2%}' for x in comparaison.proba_moyenne])
print('Top-5 :', [f'{x:.2%}' for x in comparaison.top5])
print('Différence maximale entre matrices finales :', np.max(np.abs(P_hybride_final - P_co)))

print('Matrice initiale finale (extrait 10 x 10) :')
display(pd.DataFrame(P_co, index=lieux, columns=lieux).iloc[:10, :10].round(4))
print('Matrice enrichie finale (extrait 10 x 10) :')
display(pd.DataFrame(P_hybride_final, index=lieux, columns=lieux).iloc[:10, :10].round(4))


,modele,proba_moyenne,top5,relations_evaluees,part_poids_evaluee
0,Co-visiting seul,0.046656,0.183874,4043,0.580546
1,Co-visiting + Node2Vec (liens existants),0.047581,0.174435,4043,0.580546


Probabilité moyenne : ['4.67%', '4.76%']
Top-5 : ['18.39%', '17.44%']
Différence maximale entre matrices finales : 0.06576706743135641
Matrice initiale finale (extrait 10 x 10) :


,196911,196912,196913,196916,196917,196920,196928,196931,209260,209264
196911,0.0,0.08,0.0,0.0,0.04,0.0400,0.0,0.0,0.0000,0.0000
196912,0.0,0.00,0.0,0.0,0.00,0.0476,0.0,0.0,0.0000,0.0000
196913,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
196916,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.1429,0.0000
196917,0.0,0.00,0.0,0.0,0.00,0.0588,0.0,0.0,0.0000,0.0000
196920,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
196928,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
196931,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000
209260,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.1667
209264,0.0,0.00,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.0000


Matrice enrichie finale (extrait 10 x 10) :


,196911,196912,196913,196916,196917,196920,196928,196931,209260,209264
196911,0.0,0.0856,0.0,0.0,0.0434,0.0425,0.0,0.0,0.0000,0.0000
196912,0.0,0.0000,0.0,0.0,0.0000,0.0517,0.0,0.0,0.0000,0.0000
196913,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
196916,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.1461,0.0000
196917,0.0,0.0000,0.0,0.0,0.0000,0.0639,0.0,0.0,0.0000,0.0000
196920,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
196928,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
196931,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000
209260,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.1763
209264,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.0,0.0000,0.0000


## 9 — Exporter les matrices complètes et les résultats

In [9]:
pd.DataFrame(W, index=lieux, columns=lieux).to_csv(
    DOSSIER_RESULTATS / 'matrice_poids_co_visiting_2013.csv')
pd.DataFrame(P_co, index=lieux, columns=lieux).to_csv(
    DOSSIER_RESULTATS / 'matrice_probabilites_co_visiting_2013.csv')
pd.DataFrame(P_hybride_final, index=lieux, columns=lieux).to_csv(
    DOSSIER_RESULTATS / 'matrice_probabilites_hybride_2013.csv')
resultats_validation.to_csv(DOSSIER_RESULTATS / 'validation_2014.csv', index=False)
comparaison.to_csv(DOSSIER_RESULTATS / 'comparaison_2015.csv', index=False)
comparaison_poi.to_csv(DOSSIER_RESULTATS / 'exemple_poi.csv', index=False)
print('Fichiers enregistrés dans :', DOSSIER_RESULTATS.resolve())


Fichiers enregistrés dans : C:\Users\aurel\Downloads\resultats_matrices_simples
